# 03 · Occurrence model
XGBoost multi-class → calibration (cross-fit between the two calibration years) → zone threshold → validation report split into
**Group A** (system already present) and **Group B** (no system at issue time), against persistence.

Classes: 0 none · 1 D–DD · 2 CS · 3 SCS+ within 200 km in (t0, t0+24 h].

**Runtime:** CPU + High-RAM (switch to a T4 GPU only if training is too slow: set `DEVICE = "cuda"`).
**Outputs:** `outputs/models/occurrence_*`, `outputs/results/occurrence_*.csv`, figures.

In [ ]:
# ▶ Setup: connect Google Drive, install missing packages, load the project code
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost", "lightgbm", "xgboost"], check=False)

PROJECT_DIR = "/content/drive/MyDrive/CycloneProject"      # <- change only if you put the project elsewhere
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import numpy as np, pandas as pd
from src import config as C
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("project:", C.PROJECT, "| ERA5:", C.ERA5_ROOT)

In [ ]:
import json, pickle
from src import occurrence as OC, metrics as M, plots as P, tracks
from src.baselines import occurrence_persistence_proba
from src.common import list_tables, load_table
from src.models import occurrence_feature_columns

DEVICE = "cpu"
occ_dir, full_dir = C.out("tables", "occurrence"), C.out("tables", "occurrence_full")
cols = load_table(list_tables(occ_dir)[0]).columns.tolist()
FEATS = occurrence_feature_columns(cols) + ["persistence"]   # storm already nearby -> strong signal
KEEP = FEATS + [c for c in ["lat", "lon", "time", "year", "label", "persistence", "existing_system", "w"] if c not in FEATS]
print(len(FEATS), "features")

In [ ]:
# ▶ Load training and validation years
train = OC.load_split(occ_dir, C.TRAIN, KEEP)
val = OC.load_split(occ_dir, C.VAL, KEEP)
print(f"train {len(train):,} rows ({train.memory_usage().sum()/1e9:.1f} GB) | val {len(val):,} rows")
print("train label counts:", train.label.value_counts().sort_index().to_dict())

In [ ]:
# ▶ Train XGBoost (early stopping on the validation years)
booster = OC.train_xgb(train, val, FEATS, device=DEVICE)
booster.save_model(str(C.out("models") / "occurrence_xgb.json"))
json.dump(FEATS, open(C.out("models") / "occurrence_features.json", "w"))
print("best iteration:", booster.best_iteration)
del train

## Calibration (cross-fit on the two calibration years)

In [ ]:
cal = OC.load_split(occ_dir, C.CAL, KEEP)
Mc = OC.margins(booster, cal, FEATS)
scores, best, P_oof, make_best = OC.crossfit_calibration(Mc, cal.label.values, cal.w.values, cal.year.values)
display(scores.round(4))
print("chosen calibrator:", best)
final_cal = make_best().fit(Mc, cal.label.values, cal.w.values)          # refit on both years
pickle.dump(final_cal, open(C.out("models") / "occurrence_calibrator.pkl", "wb"))
scores.to_csv(C.out("results") / "occurrence_calibration_scores.csv", index=False)

## Zone threshold on P(CS or stronger)
Uses the full calibration-year grids (every cell at 00/12 UTC). Each year is calibrated by a calibrator fitted on
the **other** year, so the threshold is chosen on out-of-sample probabilities.
Rule: highest event hit rate with at most `MAX_FALSE_ZONES` false-alarm zones per forecast.

In [ ]:
MAX_FALSE_ZONES = 0.1
THRESHOLDS = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6]
fixes = tracks.load_ibtracs(); TRK, _ = tracks.build_tracks(fixes); TP = tracks.TrackPoints(TRK)

full_cal = OC.load_split(full_dir, C.CAL, KEEP)
Mf = OC.margins(booster, full_cal, FEATS)
y1, y2 = sorted(full_cal.year.unique())
Pf = np.zeros((len(full_cal), C.N_OCC))
for fit_y, apply_y in ((y1, y2), (y2, y1)):
    a = cal.year.values == fit_y
    c = make_best().fit(Mc[a], cal.label.values[a], cal.w.values[a])
    b = full_cal.year.values == apply_y
    Pf[b] = c.predict(Mf[b])
full_cal["p_cs_plus"] = Pf[:, 2:].sum(axis=1)
ev = M.event_scores(full_cal, "p_cs_plus", TP, THRESHOLDS)
display(ev.round(3))
ok = ev[ev.false_zones_per_forecast <= MAX_FALSE_ZONES]
THR = float((ok if len(ok) else ev).sort_values(["hit_rate", "threshold"], ascending=[False, False]).iloc[0].threshold)
json.dump({"threshold_p_cs_plus": THR, "max_false_zones": MAX_FALSE_ZONES}, open(C.out("models") / "occurrence_threshold.json", "w"))
ev.to_csv(C.out("results") / "occurrence_threshold_sweep.csv", index=False)
P.threshold_curve(ev, THR, C.out("figures") / "occurrence_threshold_curve.png")
print("chosen threshold:", THR)

## Validation report (validation years), Group A / Group B vs persistence

In [ ]:
Mv = OC.margins(booster, val, FEATS)
Pv = final_cal.predict(Mv)
Pp = occurrence_persistence_proba(val.persistence.values)

def scores_for(sub_idx):
    s = {}
    y, w = val.label.values[sub_idx], val.w.values[sub_idx]
    s["logloss_model"] = M.logloss_multi(y, Pv[sub_idx], w)
    s["brier_model"] = M.brier_multi(y, Pv[sub_idx], w)
    s["brier_persistence"] = M.brier_multi(y, Pp[sub_idx], w)
    s["brier_skill_vs_persistence_%"] = 100 * (1 - s["brier_model"] / s["brier_persistence"])
    s["rows"] = int(sub_idx.sum())
    return s

groups = {"A: system present": val.existing_system.values == 1,
          "B: no system at t0": val.existing_system.values == 0,
          "all": np.ones(len(val), bool)}
rep = pd.DataFrame({g: scores_for(ix) for g, ix in groups.items()}).T
display(rep.round(4))
rep.to_csv(C.out("results") / "occurrence_validation_groups.csv")

ycs = (val.label.values >= 2).astype(int)
pcs = Pv[:, 2:].sum(axis=1)
rel = {g: M.reliability(ycs[ix], pcs[ix], val.w.values[ix]) for g, ix in groups.items() if g != "all"}
P.reliability_plot(rel, "P(CS or stronger within 200 km, 24 h) - validation", C.out("figures") / "occurrence_reliability.png")

full_val = OC.load_split(full_dir, C.VAL, KEEP)
if len(full_val):
    full_val["p_cs_plus"] = final_cal.predict(OC.margins(booster, full_val, FEATS))[:, 2:].sum(axis=1)
    evv = M.event_scores(full_val, "p_cs_plus", TP, [THR])
    display(evv.round(3)); evv.to_csv(C.out("results") / "occurrence_validation_events.csv", index=False)

In [ ]:
# ▶ Which features matter (XGBoost gain)
imp = pd.Series(booster.get_score(importance_type="gain")).sort_values(ascending=False)
imp.to_csv(C.out("results") / "occurrence_feature_importance.csv")
P.importance_bar(imp / imp.sum(), "Occurrence model - top features (gain share)", C.out("figures") / "occurrence_importance.png")
display(imp.head(15))